# Comparing Distributions Across Multiple Columns Milestone

**Pandas Fundamentals - Exploratory Data Analysis**

---

## Introduction

Comparing distributions helps you understand how different variables behave relative to each other and reveals patterns that single-column analysis cannot show.

### What You Will Learn:
- Compute summary statistics for multiple columns
- Compare means, medians, and ranges across columns
- Identify columns with higher or lower variability
- Detect unusual distributions conceptually
- Use comparisons to guide deeper analysis

### Why This Matters:
Most real insights come from **comparison**, not isolation. This milestone ensures you understand how variables differ from each other.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

print("Libraries imported successfully!")

---

## Section 1: Understanding Distributions

### What is a Distribution?

A distribution describes how values in a column are spread out.

**Key aspects:**
- **Central tendency**: Where do most values cluster? (mean, median)
- **Spread**: How far do values deviate from center? (range, std)
- **Shape**: Are values symmetric or skewed?
- **Outliers**: Are there unusual extreme values?

In [ ]:
# Create sample data with different distribution characteristics
np.random.seed(42)

sample_data = {
    'student_id': range(1, 51),
    
    # Test scores - relatively narrow range, high central tendency
    'math_score': np.random.normal(75, 8, 50).clip(50, 100),
    
    # Science scores - similar range but slightly lower mean
    'science_score': np.random.normal(70, 10, 50).clip(40, 100),
    
    # English scores - wider variability
    'english_score': np.random.normal(65, 15, 50).clip(30, 100),
    
    # Attendance percentage - very high values, low variability
    'attendance_pct': np.random.normal(92, 5, 50).clip(75, 100),
    
    # Study hours per week - lower values, moderate variability
    'study_hours': np.random.normal(15, 5, 50).clip(5, 35),
    
    # Assignment completion - percentage with moderate spread
    'assignment_completion': np.random.normal(85, 12, 50).clip(50, 100)
}

df = pd.DataFrame(sample_data)

# Round numeric columns for cleaner display
numeric_cols = ['math_score', 'science_score', 'english_score', 
                'attendance_pct', 'study_hours', 'assignment_completion']
df[numeric_cols] = df[numeric_cols].round(2)

print("✓ Sample student performance dataset created")
print(f"  - {df.shape[0]} students")
print(f"  - {len(numeric_cols)} numeric measurement columns")

In [ ]:
# Preview the data
df.head(10)

**Key observation:** Notice how different columns operate on different scales and ranges. This is exactly WHY we need to compare distributions, not just raw values.

---

## Section 2: Computing Summary Statistics

### Using describe() for Multi-Column Summary

The `describe()` method is the PRIMARY tool for comparing distributions.

**What describe() shows for each column:**
- count: Number of non-missing values
- mean: Average value (central tendency)
- std: Standard deviation (spread/variability)
- min: Minimum value
- 25%, 50%, 75%: Quartiles
- max: Maximum value

In [ ]:
# Compute summary statistics across all numeric columns
summary_stats = df[numeric_cols].describe()
summary_stats

### Interpretation Guide:

Read this table **COLUMN BY COLUMN**, then **COMPARE ACROSS** columns:

**For each column, ask:**
- What is the typical value? (mean/median)
- How much do values vary? (std)
- What is the full range? (max - min)
- Is it symmetric? (mean ≈ median) or skewed? (mean ≠ median)

**Then COMPARE columns:**
- Which has the highest average?
- Which varies the most?
- Which is most consistent?
- Do any columns behave unusually?

---

## Section 3: Comparing Central Tendency

### 3.1 Comparing Means

In [ ]:
# Calculate means for all columns
means = df[numeric_cols].mean()
print("Mean values for each column:")
print(means)
print()

# Sort to see lowest to highest
sorted_means = means.sort_values()
print("\nSorted means (lowest to highest):")
print(sorted_means)

**Key Insights:**
- Study hours has the lowest mean (different scale)
- Among test scores, which has the highest mean?
- Attendance is very high (as expected)

### 3.2 Comparing Medians

Median (middle value) is more robust to outliers than mean.

**Comparing median to mean reveals skewness:**
- If mean ≈ median: Distribution is roughly symmetric
- If mean > median: Distribution is right-skewed
- If mean < median: Distribution is left-skewed

In [ ]:
# Calculate medians
medians = df[numeric_cols].median()
print("Median values for each column:")
print(medians)

In [ ]:
# Compare mean vs median to detect skewness
comparison_df = pd.DataFrame({
    'Mean': means,
    'Median': medians,
    'Difference': means - medians,
    'Skew_Type': ['Symmetric' if abs(m - med) < 1 else 
                  ('Right-skewed' if m > med else 'Left-skewed') 
                  for m, med in zip(means, medians)]
})

comparison_df

**Why this matters:**
- Symmetric data: Mean is a good summary
- Skewed data: Median may be more representative
- Comparison reveals outlier influence

---

## Section 4: Comparing Spread and Variability

### 4.1 Comparing Standard Deviation

Standard deviation measures how spread out values are from the mean.

- **High std**: Values are widely dispersed (high variability)
- **Low std**: Values cluster tightly around the mean (low variability)

In [ ]:
# Calculate standard deviation
stds = df[numeric_cols].std()
print("Standard deviation for each column:")
print(stds)
print()

# Sort by variability
sorted_stds = stds.sort_values()
print("\nSorted by variability (lowest to highest):")
print(sorted_stds)

**Interpretation:**
- Most consistent column: Lowest standard deviation
- Most variable column: Highest standard deviation
- Low variability = predictable, stable
- High variability = diverse outcomes, less predictable

### 4.2 Comparing Ranges

Range = Maximum - Minimum

Range shows the full span of values in each column.

In [ ]:
# Calculate ranges
ranges = df[numeric_cols].max() - df[numeric_cols].min()

# Create comparison table
range_comparison = pd.DataFrame({
    'Min': df[numeric_cols].min(),
    'Max': df[numeric_cols].max(),
    'Range': ranges
})

range_comparison

### 4.3 Coefficient of Variation (Relative Variability)

**CV = (Standard Deviation / Mean) × 100**

CV shows variability RELATIVE to the mean, allowing fair comparison across different scales.

Use CV when columns have different scales (e.g., comparing test scores to study hours).

In [ ]:
# Calculate Coefficient of Variation
cv = (stds / means * 100).sort_values()

print("Coefficient of Variation (%):\n")
print(cv)
print()
print(f"✓ MOST STABLE: {cv.index[0]} (CV = {cv.iloc[0]:.2f}%)")
print(f"✓ MOST VARIABLE: {cv.index[-1]} (CV = {cv.iloc[-1]:.2f}%)")

---

## Section 5: Identifying Patterns and Anomalies

### 5.1 Comprehensive Comparison Table

In [ ]:
# Build comprehensive comparison table
comparison_table = pd.DataFrame({
    'Mean': means,
    'Median': medians,
    'Std_Dev': stds,
    'Min': df[numeric_cols].min(),
    'Max': df[numeric_cols].max(),
    'Range': ranges,
    'CV(%)': (stds / means * 100)
}).round(2)

comparison_table

### 5.2 Pattern Analysis

Look for these patterns when comparing distributions:

1. **SIMILAR DISTRIBUTIONS**: Columns with similar means and spreads
2. **CONTRASTING DISTRIBUTIONS**: Columns that differ significantly
3. **UNUSUAL VARIABILITY**: Columns with unexpectedly high/low std
4. **SCALE DIFFERENCES**: Columns operating on different ranges
5. **SKEWNESS PATTERNS**: Consistent mean-median differences

In [ ]:
# Compare test scores (similar scales)
test_scores = ['math_score', 'science_score', 'english_score']
print("Comparing Test Scores (similar scales):")
print("=" * 60)
print(comparison_table.loc[test_scores])
print()
print("Observations:")
print(f"  • Highest mean: {means[test_scores].idxmax()}")
print(f"  • Lowest mean: {means[test_scores].idxmin()}")
print(f"  • Most consistent: {stds[test_scores].idxmin()}")
print(f"  • Most variable: {stds[test_scores].idxmax()}")

In [ ]:
# Compare behavioral metrics
behavioral = ['attendance_pct', 'study_hours', 'assignment_completion']
print("Comparing Behavioral Metrics:")
print("=" * 60)
print(comparison_table.loc[behavioral])
print()
print("Observations:")
print("  • Attendance is very high and consistent")
print("  • Study hours operate on different scale")
print("  • Assignment completion similar to test scores")

### 5.3 Detecting Unusual Distributions

**Red flags:**
- Extremely high or low variability compared to others
- Mean very different from median (strong skew)
- Unexpected ranges or boundaries
- Outliers that significantly affect statistics

In [ ]:
# Detect columns with high relative variability
high_cv_threshold = 25
high_cv_cols = cv[cv > high_cv_threshold]

if len(high_cv_cols) > 0:
    print(f"Columns with HIGH relative variability (CV > {high_cv_threshold}%):")
    print(high_cv_cols)
    print("  → These columns have high diversity in values")
else:
    print("No columns with unusually high variability detected.")

print()

# Detect skewed columns
mean_median_diff = abs(means - medians)
skewed_threshold = 2
skewed_cols = mean_median_diff[mean_median_diff > skewed_threshold]

if len(skewed_cols) > 0:
    print("Columns with potential skewness (|mean - median| > 2):")
    print(skewed_cols)
    print("  → These distributions may have outliers or asymmetry")
else:
    print("No strongly skewed columns detected.")

---

## Section 6: Using Comparisons to Guide Analysis

### What Comparisons Tell You About Next Steps

Distribution comparisons directly inform analysis decisions:

**If columns have SIMILAR distributions:**
- → They might be related or measure similar constructs
- → Consider correlation analysis
- → May not need all columns (redundancy)

**If columns have DIFFERENT scales:**
- → Standardize before comparing (z-scores, normalization)
- → Use relative measures (percentiles, CV)
- → Be careful with direct aggregation

**If columns have HIGH variability:**
- → Check for outliers or data quality issues
- → Consider what drives the variation
- → May need segmentation or grouping

**If columns are SKEWED:**
- → Use median instead of mean
- → Consider transformations (log, sqrt)
- → Be cautious with parametric statistics

---

## Section 7: Practice Exercise

### Apply distribution comparison to new data

In [ ]:
# Create new dataset with different characteristics
np.random.seed(123)

sales_data = {
    'product_id': range(1, 31),
    'price': np.random.uniform(10, 200, 30),
    'cost': np.random.uniform(5, 150, 30),
    'units_sold': np.random.randint(50, 500, 30),
    'customer_rating': np.random.uniform(3.0, 5.0, 30),
    'discount_pct': np.random.uniform(0, 30, 30)
}

df_sales = pd.DataFrame(sales_data)
df_sales = df_sales.round(2)

print("New dataset: Product Sales Data")
df_sales.head()

### YOUR TASK:

Perform a complete distribution comparison analysis:

1. Compute summary statistics for all numeric columns
2. Compare central tendency (means and medians)
3. Compare variability (std dev and ranges)
4. Identify any unusual patterns or anomalies
5. Suggest next steps based on your findings

In [ ]:
# YOUR CODE HERE - Step 1: Summary statistics
sales_numeric = ['price', 'cost', 'units_sold', 'customer_rating', 'discount_pct']

# TODO: Use describe() to get summary statistics

In [ ]:
# YOUR CODE HERE - Step 2: Central tendency comparison

# TODO: Calculate means and medians, create comparison DataFrame

In [ ]:
# YOUR CODE HERE - Step 3: Variability comparison

# TODO: Calculate std dev, ranges, and CV

In [ ]:
# YOUR CODE HERE - Step 4: Identify patterns and anomalies

# TODO: Look for unusual distributions, high variability, skewness

---

### SOLUTION (Run after attempting the exercise)

In [ ]:
# Solution: Complete analysis
print("1. Summary Statistics:")
print("=" * 60)
print(df_sales[sales_numeric].describe())
print()

print("2. Central Tendency Comparison:")
print("=" * 60)
sales_means = df_sales[sales_numeric].mean()
sales_medians = df_sales[sales_numeric].median()
print(pd.DataFrame({'Mean': sales_means, 'Median': sales_medians}))
print()

print("3. Variability Comparison:")
print("=" * 60)
sales_stds = df_sales[sales_numeric].std()
sales_ranges = df_sales[sales_numeric].max() - df_sales[sales_numeric].min()
print(pd.DataFrame({'Std_Dev': sales_stds, 'Range': sales_ranges}))
print()

print("4. Key Observations:")
print("=" * 60)
print("  • Units_sold operates on much larger scale than other metrics")
print("  • Customer_rating has narrow range (3.0-5.0) as expected")
print("  • Price and cost have wide ranges, suggesting diverse products")
print("  • Discount percentages vary substantially")
print()

print("5. Suggested Next Steps:")
print("=" * 60)
print("  • Calculate profit margin (price - cost) and compare")
print("  • Analyze relationship between discount and units_sold")
print("  • Investigate if customer_rating relates to sales volume")
print("  • Segment products by price range for targeted analysis")

---

## Conclusion: Key Takeaways

### DISTRIBUTIONAL COMPARISON is essential for:
✓ Understanding how variables differ from each other  
✓ Identifying patterns and anomalies across columns  
✓ Making informed analysis decisions  
✓ Adding context to single-column statistics  

### REMEMBER:
- Compare central tendency (mean, median) to understand typical values
- Compare spread (std, range) to understand variability
- Use relative measures (CV) when scales differ
- Look for patterns, not just individual numbers
- Let comparisons guide your next analytical steps

### MOST IMPORTANTLY:
→ Always compare distributions before drawing conclusions  
→ Context from comparison prevents misleading insights  
→ Multi-column thinking reveals relationships  
→ Distribution analysis is the foundation of good EDA  

---

## Next Steps:
- Practice with your own datasets
- Try comparing distributions in different domains
- Learn correlation analysis to quantify relationships
- Explore visualization for distribution comparison